# Radiology image–caption pair verification

The experiment reuses frozen CLIP embeddings for 200 training and 80 official test pairs from the Radiology_mini subset of ROCOv2. Five 150/50 train/validation partitions select ridge strength and blend weight by average precision, then the maps refit on all 200 training pairs. Each test grid contains 80 positives and 6,320 off-diagonal negatives. All five fits share one test gallery. Blur and word dropout provide three stress conditions.

Published result: On clean pairs, the trained blend raised AUROC from 0.855 to 0.873 and bidirectional Recall@5 from 48.1% to 52.3%. Average precision fell from 0.118 to 0.108. Pair labels identify the original caption. Different captions can still describe clinically similar images.

In [ ]:
from pathlib import Path
import subprocess, sys
root = Path.cwd() / 'roco-pair-verification-real-data-run'
root.mkdir(exist_ok=True)
subprocess.run([sys.executable, "-m", "pip", "install", 'numpy==2.2.6', 'scikit-learn==1.7.2', 'torch==2.14.1', 'transformers==5.20.2', 'datasets==5.2.0', 'Pillow==12.3.0'], check=True)

### prepare.py

In [ ]:
(root / 'prepare.py').write_text('from __future__ import annotations\n\n\nimport csv\nimport json\nimport os\nimport random\n\n\nfrom itertools import islice\nfrom pathlib import Path\n\nos.environ.setdefault(\n    "HF_HOME", str(Path(__file__).resolve().parent / "data" / "hf_cache")\n)\nos.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")\n\nimport numpy as np\nimport torch\n\nimport torch.nn.functional as F\nfrom datasets import load_dataset\nfrom PIL import ImageFilter\nfrom transformers import AutoProcessor, CLIPModel\n\nROOT = Path(__file__).resolve().parent\n\nDATA_DIR = ROOT / "data"\nEMBEDDING_PATH = DATA_DIR / "roco_clip_embeddings.npz"\nMODEL_NAME = "openai/clip-vit-base-patch32"\nTRAIN_COUNT = 200\nTEST_COUNT = 80\nBATCH_SIZE = 16\nDEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n\n\ndef collect(split, count):\n    stream = load_dataset(\n        "unsloth/Radiology_mini",\n        split=split,\n        streaming=True,\n        revision="12468bc1f3cecf1056c290182a0e88ce506049f9",\n    )\n    rows = []\n    for example in islice(stream, count):\n        rows.append(\n            {\n                "image": example["image"].convert("RGB").copy(),\n                "image_id": example["image_id"],\n                "caption": example["caption"],\n            }\n        )\n    if len(rows) != count:\n        raise RuntimeError(f"Expected {count} {split} rows, received {len(rows)}")\n    return rows\n\n\ndef dropped_caption(caption, seed):\n    words = caption.split()\n    if len(words) <= 3:\n        return caption\n    rng = random.Random(seed)\n    kept = [word for word in words if rng.random() > 0.35]\n    return " ".join(kept if len(kept) >= 3 else words[:3])\n\n\ndef embed_images(model, processor, images):\n    outputs = []\n    for start in range(0, len(images), BATCH_SIZE):\n        batch = processor(\n            images=images[start : start + BATCH_SIZE], return_tensors="pt"\n        )\n        batch = {key: value.to(DEVICE) for key, value in batch.items()}\n        with torch.no_grad():\n            features = model.get_image_features(**batch)\n            if hasattr(features, "pooler_output"):\n                features = features.pooler_output\n        outputs.append(F.normalize(features, dim=1).cpu())\n        print(\n            json.dumps(\n                {\n                    "embedded_images": min(start + BATCH_SIZE, len(images)),\n                    "total": len(images),\n                }\n            ),\n            flush=True,\n        )\n    return torch.cat(outputs).numpy().astype(np.float32)\n\n\ndef embed_text(model, processor, captions):\n    outputs = []\n    for start in range(0, len(captions), BATCH_SIZE * 2):\n        batch = processor(\n            text=captions[start : start + BATCH_SIZE * 2],\n            padding=True,\n            truncation=True,\n            return_tensors="pt",\n        )\n        batch = {key: value.to(DEVICE) for key, value in batch.items()}\n        with torch.no_grad():\n            features = model.get_text_features(**batch)\n            if hasattr(features, "pooler_output"):\n                features = features.pooler_output\n        outputs.append(F.normalize(features, dim=1).cpu())\n    return torch.cat(outputs).numpy().astype(np.float32)\n\n\ndef prepare_embeddings():\n    if EMBEDDING_PATH.exists():\n        return dict(np.load(EMBEDDING_PATH))\n    DATA_DIR.mkdir(parents=True, exist_ok=True)\n    train_rows = collect("train", TRAIN_COUNT)\n    test_rows = collect("test", TEST_COUNT)\n    with (DATA_DIR / "sample_manifest.csv").open(\n        "w", newline="", encoding="utf-8"\n    ) as handle:\n        writer = csv.writer(handle)\n        writer.writerow(["split", "image_id", "caption"])\n        writer.writerows(\n            [["train", row["image_id"], row["caption"]] for row in train_rows]\n        )\n        writer.writerows(\n            [["test", row["image_id"], row["caption"]] for row in test_rows]\n        )\n    processor = AutoProcessor.from_pretrained(\n        MODEL_NAME, revision="3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268"\n    )\n    model = (\n        CLIPModel.from_pretrained(\n            MODEL_NAME, revision="3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268"\n        )\n        .to(DEVICE)\n        .eval()\n    )\n    train_images = [row["image"] for row in train_rows]\n    test_images = [row["image"] for row in test_rows]\n    train_captions = [row["caption"] for row in train_rows]\n    test_captions = [row["caption"] for row in test_rows]\n    arrays = {\n        "train_image": embed_images(model, processor, train_images),\n        "train_text": embed_text(model, processor, train_captions),\n        "test_image": embed_images(model, processor, test_images),\n        "test_text": embed_text(model, processor, test_captions),\n        "test_blur_image": embed_images(\n            model,\n            processor,\n            [image.filter(ImageFilter.GaussianBlur(3.0)) for image in test_images],\n        ),\n        "test_dropout_text": embed_text(\n            model,\n            processor,\n            [\n                dropped_caption(caption, 9000 + index)\n                for index, caption in enumerate(test_captions)\n            ],\n        ),\n    }\n    np.savez_compressed(EMBEDDING_PATH, **arrays)\n    return arrays\n\n\nif __name__ == "__main__":\n    prepare_embeddings()\n', encoding="utf8")

### train.py

In [ ]:
(root / 'train.py').write_text('"""Train an image-caption compatibility score on paired radiology embeddings."""\n\nimport argparse\nfrom pathlib import Path\nimport hashlib\nimport json\nimport numpy as np\nfrom sklearn.metrics import roc_auc_score, average_precision_score\nfrom sklearn.model_selection import train_test_split\n\nROOT = Path(__file__).resolve().parent\n\n\ndef unit(x):\n    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-10)\n\n\ndef metrics(scores):\n    labels = np.eye(len(scores), dtype=bool)\n    # All gallery negatives are included. Each genuine pair contributes one positive.\n    return {\n        "pair_auroc": roc_auc_score(labels.ravel(), scores.ravel()),\n        "average_precision": average_precision_score(labels.ravel(), scores.ravel()),\n        "recall_at_5": np.mean(\n            [\n                np.any(\n                    np.argsort(-m, axis=1)[:, :5] == np.arange(len(m))[:, None], axis=1\n                ).mean()\n                for m in (scores, scores.T)\n            ]\n        ),\n    }\n\n\nclass Compatibility:\n    def __init__(self, images, captions, ridge):\n        x, t = unit(images), unit(captions)\n        self.mx, self.mt = x.mean(0), t.mean(0)\n        xc, tc = x - self.mx, t - self.mt\n        # Solve in the smaller sample space. Learn both directions from paired rows.\n        self.xt = xc.T @ np.linalg.solve(xc @ xc.T + ridge * np.eye(len(x)), tc)\n        self.tx = tc.T @ np.linalg.solve(tc @ tc.T + ridge * np.eye(len(t)), xc)\n\n    def score(self, x, t):\n        x, t = unit(x) - self.mx, unit(t) - self.mt\n        return (unit(x @ self.xt) @ unit(t).T + unit(x) @ unit(t @ self.tx).T) / 2\n\n\ndef run(a, seed):\n    x, t = a["train_image"], a["train_text"]\n    fit, val = train_test_split(np.arange(len(x)), test_size=0.25, random_state=seed)\n    cos = unit(x[val]) @ unit(t[val]).T\n    candidates = []\n    for ridge in (0.05, 0.5, 5.0):\n        model = Compatibility(x[fit], t[fit], ridge)\n        learned = model.score(x[val], t[val])\n        for weight in (0.0, 0.1, 0.3, 0.6, 1.0):\n            score = (1 - weight) * cos + weight * learned\n            candidates.append((metrics(score)["average_precision"], weight, ridge))\n    _, weight, ridge = max(candidates, key=lambda row: row[0])\n    model = Compatibility(x, t, ridge)\n    dest = ROOT / "checkpoints"\n    dest.mkdir(exist_ok=True)\n    np.savez_compressed(\n        dest / f"blend-{seed}.npz",\n        image_mean=model.mx,\n        text_mean=model.mt,\n        image_to_text=model.xt,\n        text_to_image=model.tx,\n        blend_weight=weight,\n        ridge=ridge,\n    )\n    result = {}\n    for tag, ik, tk in [\n        ("clean", "test_image", "test_text"),\n        ("blur", "test_blur_image", "test_text"),\n        ("dropout", "test_image", "test_dropout_text"),\n        ("both", "test_blur_image", "test_dropout_text"),\n    ]:\n        cos = unit(a[ik]) @ unit(a[tk]).T\n        pred = model.score(a[ik], a[tk])\n        for method, s in [\n            ("cosine", cos),\n            ("ridge", pred),\n            ("blend", (1 - weight) * cos + weight * pred),\n        ]:\n            result[f"{method}_{tag}"] = metrics(s)\n    return {"validation_weight": weight, "validation_ridge": ridge, "test": result}\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\n        "--embeddings",\n        type=Path,\n        default=ROOT / "data/roco_clip_embeddings.npz",\n    )\n    path = parser.parse_args().embeddings\n    with np.load(path) as a:\n        assert a["train_image"].shape == a["train_text"].shape == (200, 512)\n        assert a["test_image"].shape == a["test_text"].shape == (80, 512)\n        runs = {s: run(a, s) for s in [17, 29, 43, 59, 71]}\n    means = {\n        k: {m: np.mean([r["test"][k][m] for r in runs.values()]) for m in v}\n        for k, v in runs[17]["test"].items()\n    }\n    (ROOT / "results.json").write_text(\n        json.dumps(\n            {\n                "embedding_sha256": hashlib.sha256(path.read_bytes()).hexdigest(),\n                "training_pairs": 200,\n                "test_pairs": 80,\n                "runs": runs,\n                "means": means,\n            },\n            indent=2,\n        )\n    )\n', encoding="utf8")

### Train and evaluate

In [ ]:
subprocess.run([sys.executable, 'prepare.py'], cwd=root, check=True)
subprocess.run([sys.executable, 'train.py', '--embeddings', 'data/roco_clip_embeddings.npz'], cwd=root, check=True)

In [ ]:
import json
report = json.loads((root / "results.json").read_text())
print(json.dumps(report.get("means", report.get("subject_macro")), indent=2))